In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DoubleType, TimestampType, DateType

catalog_name="ecommerce"

df=spark.read.table(f"{catalog_name}.bronze.brz_order_items")

display(df.limit(3))

In [0]:
df.printSchema()

remove duplicates- discrepancy data

In [0]:
df=df.dropDuplicates(["order_id","item_seq"])

#convert two to 2 and convert to int in column quantity
#normal casting- since spark is lazy it's casting before converting two to 2. so extra step


df.select("quantity").distinct().show()

df=df.withColumn("quantity"
                 ,F.when(F.col("quantity")=="Two",2).otherwise(F.col("quantity"))).withColumn\
                 ("quantity",F.col("quantity").cast("int")
                             )

#remove $ or any symbol from unit price column

df=df.withColumn("unit_price",
                 F.regexp_replace("unit_price","[$]","").cast("double")
                 )

#remove % from discount_pct column

df=df.withColumn("discount_pct",
                 F.regexp_replace("discount_pct","%","").cast("double"))

#covert coupon code to lower

df=df.withColumn("coupon_code",F.lower(F.trim(F.col("coupon_code"))))

#more neat look of channel column
#check distint channels
df.select("channel").distinct().show()

df=df.withColumn("channel",
                 F.when(F.col("channel") == "web", "Website")
                 .when(F.col("channel") ==  "app", "Mobile")
                 .otherwise(F.col("channel")),
                 )

display(df.limit(4))


datetype conversions

In [0]:
#convert dt to datetype

df=df.withColumn("dt",F.to_date(F.col("dt"),"yyyy-MM-dd"))

#convert order_ts to timestamp

df=df.withColumn("order_ts",
                 F.coalesce(
                     F.to_timestamp(F.col("order_ts"),"yyyy-MM-dd HH:mm:ss"), #matches 2025-08-01 22:53:52
                     F.to_timestamp(F.col("order_ts"),"dd-MM-yyyy HH:mm")     #fallback for 01-08-2025 22:53
                 )
                 )


#convert item_seq to int

df=df.withColumn("item_seq",F.col("item_seq").cast("int"))

#convert tax_amount

df=df.withColumn("tax_amount",F.regexp_replace("tax_amount",r"[^0-9.\-]","").cast("double"))

#add processed time

df=df.withColumn("processed_time",F.current_timestamp())

display(df.limit(4))

In [0]:
df.printSchema()

In [0]:
df.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.silver.slv_order_items")